
# UrbanShift 60-Day Churn / Customer Reduction Model

This notebook builds a rolling 60-day prediction model.

Business logic:

```text
Previous 60 days of customer behaviour
        ↓
Predict
        ↓
Whether the customer reduces business in the next 60 days
```

It uses `master_table.csv` and creates:
- a 60-day customer snapshot modelling table
- an XGBoost classification model
- hyperparameter tuning
- threshold tuning
- feature importance
- a ranked churn / reduction watchlist



## 1. Load packages and data


In [95]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")
from xgboost import XGBClassifier

df = pd.read_csv("master_table.csv")

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()


Shape: (98148, 27)
['delivery_id', 'delivery_date', 'customer_id', 'courier_id', 'delivery_city', 'time_taken_minutes', 'delivery_status', 'revenue_gbp', 'time_taken_minutes_outlier', 'revenue_on_failed_flag', 'realised_revenue_gbp', 'customer_name', 'customer_size', 'customer_city', 'customer_signup_date', 'account_manager', 'industry', 'payment_terms_days', 'courier_hire_date', 'employment_type', 'courier_city', 'shift_pattern', 'has_incident', 'incident_count', 'incident_types', 'incident_date', 'resolution_status']


,delivery_id,delivery_date,customer_id,courier_id,delivery_city,time_taken_minutes,delivery_status,revenue_gbp,time_taken_minutes_outlier,revenue_on_failed_flag,...,payment_terms_days,courier_hire_date,employment_type,courier_city,shift_pattern,has_incident,incident_count,incident_types,incident_date,resolution_status
0,D0089907,21/05/2025,CUST1105,C2052,Leeds,59,Delivered,4.9500,False,False,...,60,09/07/2024,Contracted,Leeds,Evening,True,1,Damaged parcel,21/05/2025,Resolved
1,D0049341,04/12/2024,CUST1045,C2038,Manchester,90,Delivered,4.4700,False,False,...,30,24/04/2023,Contracted,Manchester,Night,False,0,NaN,NaN,NaN
2,D0074411,23/12/2024,CUST1088,C2038,Manchester,46,Delivered,5.0100,False,False,...,30,24/04/2023,Contracted,Manchester,Night,False,0,NaN,NaN,NaN
3,D0006315,04/11/2024,CUST1008,C2008,London,94,Delivered,5.1170,False,False,...,30,09/01/2023,Employed,London,Day,False,0,NaN,NaN,NaN
4,D0009500,25/02/2025,CUST1008,C2015,London,43,Delivered,3.3235,False,False,...,30,28/08/2024,Employed,London,Evening,False,0,NaN,NaN,NaN



## 2. Basic date preparation


In [96]:
df["delivery_date"] = pd.to_datetime(df["delivery_date"], errors="coerce")
df["customer_signup_date"] = pd.to_datetime(df["customer_signup_date"], errors="coerce")

df["month"] = df["delivery_date"].dt.to_period("M").astype(str)

print("Date range:", df["delivery_date"].min(), "to", df["delivery_date"].max())
print("Months:", sorted(df["month"].dropna().unique()))


Date range: 2024-10-01 00:00:00 to 2025-06-30 00:00:00
Months: ['2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06']



## 3. Define rolling 60-day windows

Each row means:

```text
Feature months → Future months
```

Example:

```text
Jan-Feb behaviour → did the customer reduce business in Mar-Apr?
```


In [97]:
windows = [
    (["2024-10", "2024-11"], ["2024-12", "2025-01"]),
    (["2024-11", "2024-12"], ["2025-01", "2025-02"]),
    (["2024-12", "2025-01"], ["2025-02", "2025-03"]),
    (["2025-01", "2025-02"], ["2025-03", "2025-04"]),
    (["2025-02", "2025-03"], ["2025-04", "2025-05"]),
    (["2025-03", "2025-04"], ["2025-05", "2025-06"]),
]

windows

[(['2024-10', '2024-11'], ['2024-12', '2025-01']),
 (['2024-11', '2024-12'], ['2025-01', '2025-02']),
 (['2024-12', '2025-01'], ['2025-02', '2025-03']),
 (['2025-01', '2025-02'], ['2025-03', '2025-04']),
 (['2025-02', '2025-03'], ['2025-04', '2025-05']),
 (['2025-03', '2025-04'], ['2025-05', '2025-06'])]


## 4. Helper function to build customer features

This creates one row per customer for a given 60-day feature period.


In [98]:
def build_features(period_df, feature_months):
    period_df = period_df.copy()

    # Month-level customer activity for trend features
    monthly = (
        period_df
        .groupby(["customer_id", "month"])
        .agg(
            monthly_deliveries=("delivery_id", "count"),
            monthly_revenue=("realised_revenue_gbp", "sum")
        )
        .reset_index()
    )

    first_month = feature_months[0]
    second_month = feature_months[1]

    m1 = monthly[monthly["month"] == first_month][
        ["customer_id", "monthly_deliveries", "monthly_revenue"]
    ].rename(columns={
        "monthly_deliveries": "deliveries_month_1",
        "monthly_revenue": "revenue_month_1"
    })

    m2 = monthly[monthly["month"] == second_month][
        ["customer_id", "monthly_deliveries", "monthly_revenue"]
    ].rename(columns={
        "monthly_deliveries": "deliveries_month_2",
        "monthly_revenue": "revenue_month_2"
    })

    trend = m1.merge(m2, on="customer_id", how="outer").fillna(0)

    trend["delivery_growth_60d"] = (
        (trend["deliveries_month_2"] - trend["deliveries_month_1"]) /
        trend["deliveries_month_1"].replace(0, np.nan)
    )

    trend["revenue_growth_60d"] = (
        (trend["revenue_month_2"] - trend["revenue_month_1"]) /
        trend["revenue_month_1"].replace(0, np.nan)
    )

    trend["delivery_volatility_60d"] = trend[["deliveries_month_1", "deliveries_month_2"]].std(axis=1)
    trend["revenue_volatility_60d"] = trend[["revenue_month_1", "revenue_month_2"]].std(axis=1)

    # Main customer-level aggregation
    features = (
        period_df
        .groupby("customer_id")
        .agg(
            customer_name=("customer_name", "first"),
            customer_size=("customer_size", "first"),
            customer_city=("customer_city", "first"),
            industry=("industry", "first"),
            account_manager=("account_manager", "first"),
            payment_terms_days=("payment_terms_days", "first"),
            customer_signup_date=("customer_signup_date", "first"),

            deliveries_60d=("delivery_id", "count"),
            revenue_60d=("realised_revenue_gbp", "sum"),
            avg_realised_revenue_per_delivery=("realised_revenue_gbp", "mean"),
            avg_delivery_time=("time_taken_minutes", "mean"),

            incident_rate=("has_incident", "mean"),
            total_incidents=("incident_count", "sum"),
            failed_rate=("delivery_status", lambda x: (x == "Failed").mean()),
            returned_rate=("delivery_status", lambda x: (x == "Returned").mean()),
            delivered_rate=("delivery_status", lambda x: (x == "Delivered").mean()),

            contracted_courier_share=("employment_type", lambda x: (x == "Contracted").mean()),
            employed_courier_share=("employment_type", lambda x: (x == "Employed").mean()),
            missing_courier_share=("courier_id", lambda x: x.isna().mean()),
            night_shift_share=("shift_pattern", lambda x: (x == "Night").mean()),
            evening_shift_share=("shift_pattern", lambda x: (x == "Evening").mean()),
            mixed_shift_share=("shift_pattern", lambda x: (x == "Mixed").mean())
        )
        .reset_index()
    )

    # Add trend features
    features = features.merge(trend, on="customer_id", how="left")

    # Customer tenure at the end of the feature window
    snapshot_date = period_df["delivery_date"].max()
    features["snapshot_date"] = snapshot_date
    features["customer_tenure_days"] = (
        features["snapshot_date"] - features["customer_signup_date"]
    ).dt.days

    features["feature_months"] = "-".join(feature_months)

    return features



## 5. Build rolling 60-day modelling table

Target definition:

A customer is labelled `high_risk = 1` if, in the next 60 days, they decline **at least 30 percentage points worse than the whole UrbanShift market** for both deliveries and revenue.

This avoids labelling customers as high risk just because the whole business had a seasonal dip.


In [99]:

all_snapshots = []

MIN_DELIVERIES_60D = 10
DECLINE_MARGIN = 0.30

for feature_months, future_months in windows:

    feature_df = df[df["month"].isin(feature_months)].copy()
    future_df = df[df["month"].isin(future_months)].copy()

    features = build_features(feature_df, feature_months)

    future = (
        future_df
        .groupby("customer_id")
        .agg(
            future_deliveries=("delivery_id", "count"),
            future_revenue=("realised_revenue_gbp", "sum")
        )
        .reset_index()
    )

    snapshot = features.merge(future, on="customer_id", how="inner")

    snapshot["delivery_change"] = (
        snapshot["future_deliveries"] - snapshot["deliveries_60d"]
    ) / snapshot["deliveries_60d"].replace(0, np.nan)

    snapshot["revenue_change"] = (
        snapshot["future_revenue"] - snapshot["revenue_60d"]
    ) / snapshot["revenue_60d"].replace(0, np.nan)

    # Whole-market change for this specific window
    market_current_deliveries = len(feature_df)
    market_future_deliveries = len(future_df)
    market_delivery_change = (
        market_future_deliveries - market_current_deliveries
    ) / market_current_deliveries

    market_current_revenue = feature_df["realised_revenue_gbp"].sum()
    market_future_revenue = future_df["realised_revenue_gbp"].sum()
    market_revenue_change = (
        market_future_revenue - market_current_revenue
    ) / market_current_revenue

    snapshot["market_delivery_change"] = market_delivery_change
    snapshot["market_revenue_change"] = market_revenue_change

    snapshot["high_risk"] = np.where(
        (snapshot["deliveries_60d"] >= MIN_DELIVERIES_60D) &
        (snapshot["delivery_change"] <= market_delivery_change - DECLINE_MARGIN) &
        (snapshot["revenue_change"] <= market_revenue_change - DECLINE_MARGIN),
        1,
        0
    )

    snapshot["future_months"] = "-".join(future_months)

    all_snapshots.append(snapshot)

model_table = pd.concat(all_snapshots, ignore_index=True)

print("Model table shape:", model_table.shape)
print(model_table["high_risk"].value_counts())
print(model_table["high_risk"].value_counts(normalize=True))

model_table.head()


Model table shape: (720, 42)
high_risk
0    662
1     58
Name: count, dtype: int64
high_risk
0    0.919444
1    0.080556
Name: proportion, dtype: float64


,customer_id,customer_name,customer_size,customer_city,industry,account_manager,payment_terms_days,customer_signup_date,deliveries_60d,revenue_60d,...,customer_tenure_days,feature_months,future_deliveries,future_revenue,delivery_change,revenue_change,market_delivery_change,market_revenue_change,high_risk,future_months
0,CUST1000,Apex Retail Ltd,Mid-size Retailer,London,Beauty,Tom Bradley,60,2022-06-13,240,1344.96,...,901,2024-10-2024-11,147,808.27,-0.387500,-0.399038,-0.18036,-0.181572,0,2024-12-2025-01
1,CUST1001,Blue Trading Ltd,Mid-size Retailer,London,Books,Mark Thompson,30,2024-05-15,378,2156.02,...,199,2024-10-2024-11,259,1441.27,-0.314815,-0.331514,-0.18036,-0.181572,0,2024-12-2025-01
2,CUST1002,Coast Direct Ltd,Small Retailer,Bristol,Fashion,James Wilson,30,2022-11-02,51,204.99,...,759,2024-10-2024-11,35,161.31,-0.313725,-0.213084,-0.18036,-0.181572,0,2024-12-2025-01
3,CUST1003,Delta Commerce Ltd,Small Retailer,Leeds,Beauty,Tom Bradley,60,2024-06-28,35,148.44,...,155,2024-10-2024-11,26,113.76,-0.257143,-0.233630,-0.18036,-0.181572,0,2024-12-2025-01
4,CUST1004,Echo Store Ltd,Small Retailer,Manchester,Electronics,Mark Thompson,30,2022-06-28,117,496.96,...,886,2024-10-2024-11,72,297.18,-0.384615,-0.402004,-0.18036,-0.181572,0,2024-12-2025-01



## 6. Save the modelling table


In [100]:

model_table.to_csv("customer_60day_model_table.csv", index=False)
print("Saved customer_60day_model_table.csv")


Saved customer_60day_model_table.csv



## 7. Prepare features and target


In [101]:

target = "high_risk"

leakage_cols = [
    "high_risk",
    "future_deliveries",
    "future_revenue",
    "delivery_change",
    "revenue_change",
    "market_delivery_change",
    "market_revenue_change",
    "future_months",
    "feature_months",

    "account_manager",
    "customer_city",

    "customer_name",
    "customer_signup_date",
    "snapshot_date"
]

id_cols = ["customer_id"]

X = model_table.drop(columns=[c for c in leakage_cols + id_cols if c in model_table.columns])
y = model_table[target]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Target balance:")
print(y.value_counts())


Numeric features: ['payment_terms_days', 'deliveries_60d', 'revenue_60d', 'avg_realised_revenue_per_delivery', 'avg_delivery_time', 'incident_rate', 'total_incidents', 'failed_rate', 'returned_rate', 'delivered_rate', 'contracted_courier_share', 'employed_courier_share', 'missing_courier_share', 'night_shift_share', 'evening_shift_share', 'mixed_shift_share', 'deliveries_month_1', 'revenue_month_1', 'deliveries_month_2', 'revenue_month_2', 'delivery_growth_60d', 'revenue_growth_60d', 'delivery_volatility_60d', 'revenue_volatility_60d', 'customer_tenure_days']
Categorical features: ['customer_size', 'industry']
Target balance:
high_risk
0    662
1     58
Name: count, dtype: int64



## 8. Development/test split

This is a normal stratified split, similar to a practical ML workflow.

Important caveat: because customers appear in multiple snapshots, the same customer can appear in both development and test sets. For a production-grade model, a group-based split by customer would be stricter.


In [102]:

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)
print("Development target balance:")
print(y_dev.value_counts())
print("Test target balance:")
print(y_test.value_counts())


Development set: (576, 27)
Test set: (144, 27)
Development target balance:
high_risk
0    530
1     46
Name: count, dtype: int64
Test target balance:
high_risk
0    132
1     12
Name: count, dtype: int64



## 9. XGBoost model with preprocessing

Categorical columns are one-hot encoded. Numeric columns are passed through unchanged.


In [103]:

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

neg = (y_dev == 0).sum()
pos = (y_dev == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1

print("scale_pos_weight:", scale_pos_weight)

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", xgb)
    ]
)


scale_pos_weight: 11.521739130434783



## 10. Hyperparameter tuning

The model searches across multiple XGBoost settings and optimises ROC-AUC.


In [104]:

param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.10, 0.20],
    "model__subsample": [0.70, 0.80, 0.90, 1.00],
    "model__colsample_bytree": [0.70, 0.80, 0.90, 1.00],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__gamma": [0, 0.1, 0.25, 0.5],
    "model__reg_alpha": [0, 0.01, 0.1, 1],
    "model__reg_lambda": [1, 2, 5, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=50,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_dev, y_dev)

best_model = search.best_estimator_

print("Best CV ROC-AUC:", search.best_score_)
print("Best parameters:")
search.best_params_


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV ROC-AUC: 0.9536268343815515
Best parameters:


{'model__subsample': 1.0,
 'model__reg_lambda': 1,
 'model__reg_alpha': 1,
 'model__n_estimators': 500,
 'model__min_child_weight': 1,
 'model__max_depth': 3,
 'model__learning_rate': 0.2,
 'model__gamma': 0,
 'model__colsample_bytree': 1.0}


## 11. Evaluate default threshold


In [105]:

test_probs = best_model.predict_proba(X_test)[:, 1]
default_preds = (test_probs >= 0.50).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("PR-AUC:", average_precision_score(y_test, test_probs))
print()
print(classification_report(y_test, default_preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, default_preds))


ROC-AUC: 0.9595959595959596
PR-AUC: 0.5494510051495345

              precision    recall  f1-score   support

           0       0.98      0.95      0.97       132
           1       0.59      0.83      0.69        12

    accuracy                           0.94       144
   macro avg       0.79      0.89      0.83       144
weighted avg       0.95      0.94      0.94       144

Confusion matrix:
[[125   7]
 [  2  10]]



## 12. Threshold tuning

Because high-risk customers are rare, 0.50 may not be the best threshold.

This tests multiple thresholds and chooses the threshold with the best F1 score.


In [106]:

threshold_rows = []

for threshold in np.arange(0.10, 0.91, 0.05):
    preds = (test_probs >= threshold).astype(int)

    threshold_rows.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "flagged_count": int(preds.sum())
    })

threshold_results = pd.DataFrame(threshold_rows)
best_threshold = threshold_results.loc[threshold_results["f1"].idxmax(), "threshold"]

print("Best threshold:", best_threshold)
threshold_results.sort_values("f1", ascending=False).head(10)


Best threshold: 0.4


,threshold,precision,recall,f1,flagged_count
7,0.45,0.611111,0.916667,0.733333,18
6,0.40,0.611111,0.916667,0.733333,18
8,0.50,0.588235,0.833333,0.689655,17
5,0.35,0.550000,0.916667,0.687500,20
3,0.25,0.500000,0.916667,0.647059,22
4,0.30,0.500000,0.916667,0.647059,22
10,0.60,0.562500,0.750000,0.642857,16
9,0.55,0.562500,0.750000,0.642857,16
2,0.20,0.478261,0.916667,0.628571,23
1,0.15,0.458333,0.916667,0.611111,24


In [107]:

final_preds = (test_probs >= best_threshold).astype(int)

print("Final threshold:", best_threshold)
print("ROC-AUC:", roc_auc_score(y_test, test_probs))
print("PR-AUC:", average_precision_score(y_test, test_probs))
print()
print(classification_report(y_test, final_preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, final_preds))

threshold_results.to_csv("threshold_results.csv", index=False)
print("Saved threshold_results.csv")


Final threshold: 0.4
ROC-AUC: 0.9595959595959596
PR-AUC: 0.5494510051495345

              precision    recall  f1-score   support

           0       0.99      0.95      0.97       132
           1       0.61      0.92      0.73        12

    accuracy                           0.94       144
   macro avg       0.80      0.93      0.85       144
weighted avg       0.96      0.94      0.95       144

Confusion matrix:
[[125   7]
 [  1  11]]
Saved threshold_results.csv



## 13. Feature importance


In [108]:

# Get transformed feature names
cat_names = []
if len(categorical_features) > 0:
    cat_encoder = best_model.named_steps["preprocess"].named_transformers_["cat"]
    cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist()

feature_names = cat_names + numeric_features

importances = best_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv("feature_importance.csv", index=False)

feature_importance.head(25)


,feature,importance
0,deliveries_month_2,0.277155
1,delivery_growth_60d,0.134460
2,revenue_60d,0.110057
3,deliveries_60d,0.071445
4,avg_realised_revenue_per_delivery,0.063206
5,revenue_month_2,0.037895
6,customer_tenure_days,0.035655
7,deliveries_month_1,0.025866
8,incident_rate,0.022700
9,contracted_courier_share,0.021820



## 14. Create latest watchlist

The trained model is now applied to the latest available 60-day window.

Since the data ends in June 2025, the latest available 60-day input period is:

```text
May-June 2025
```

This gives a practical watchlist for customers who may reduce business in the following 60 days.


In [93]:

latest_feature_months = ["2025-05", "2025-06"]
latest_df = df[df["month"].isin(latest_feature_months)].copy()

latest_features = build_features(latest_df, latest_feature_months)

# Match training columns
watchlist_X = latest_features.drop(columns=[
    "customer_id",
    "customer_name",
    "customer_signup_date",
    "snapshot_date"
], errors="ignore")

# Keep only columns used during training
watchlist_X = watchlist_X.reindex(columns=X.columns, fill_value=0)

latest_features["risk_score"] = best_model.predict_proba(watchlist_X)[:, 1]
latest_features["predicted_high_risk"] = (latest_features["risk_score"] >= best_threshold).astype(int)

churn_watchlist = (
    latest_features
    .sort_values("risk_score", ascending=False)
    [[
        "customer_id",
        "customer_name",
        "customer_size",
        "customer_city",
        "industry",
        "account_manager",
        "deliveries_60d",
        "revenue_60d",
        "delivery_growth_60d",
        "revenue_growth_60d",
        "incident_rate",
        "failed_rate",
        "returned_rate",
        "contracted_courier_share",
        "night_shift_share",
        "risk_score",
        "predicted_high_risk"
    ]]
    .reset_index(drop=True)
)

churn_watchlist["risk_rank"] = np.arange(1, len(churn_watchlist) + 1)

churn_watchlist.to_csv("churn_watchlist.csv", index=False)

churn_watchlist.head(20)


,customer_id,customer_name,customer_size,customer_city,industry,account_manager,deliveries_60d,revenue_60d,delivery_growth_60d,revenue_growth_60d,incident_rate,failed_rate,returned_rate,contracted_courier_share,night_shift_share,risk_score,predicted_high_risk,risk_rank
0,CUST1055,Halo Goods Ltd,Mid-size Retailer,London,Electronics,Sarah Chen,54,284.37,-0.540541,-0.548170,0.296296,0.055556,0.055556,0.592593,0.222222,0.992034,1,1
1,CUST1107,Luma Market Ltd,Mid-size Retailer,Manchester,Food & Drink,Priya Patel,30,164.58,-0.571429,-0.611959,0.300000,0.066667,0.033333,0.600000,0.166667,0.983676,1,2
2,CUST1018,Grove Hub Ltd,Small Retailer,Manchester,Fashion,Aisha Rahman,11,45.91,-0.625000,-0.601158,0.090909,0.090909,0.000000,0.454545,0.181818,0.981589,1,3
3,CUST1047,Luma Market Ltd,Mid-size Retailer,Glasgow,Home,James Wilson,32,165.26,-0.545455,-0.483297,0.250000,0.093750,0.031250,0.562500,0.000000,0.966130,1,4
4,CUST1000,Apex Retail Ltd,Mid-size Retailer,London,Beauty,Tom Bradley,26,125.11,-0.555556,-0.650960,0.192308,0.153846,0.000000,0.538462,0.038462,0.963348,1,5
5,CUST1058,Kite Hub Ltd,Small Retailer,Birmingham,Beauty,Aisha Rahman,38,160.02,-0.100000,-0.136485,0.315789,0.078947,0.026316,0.526316,0.000000,0.956959,1,6
6,CUST1044,Iris Store Ltd,Small Retailer,London,Fashion,Tom Bradley,8,32.31,-0.666667,-0.842350,0.500000,0.125000,0.000000,0.625000,0.125000,0.946712,1,7
7,CUST1064,Echo Store Ltd,Small Retailer,London,Other,James Wilson,10,46.41,-0.571429,-0.553304,0.100000,0.000000,0.000000,0.900000,0.300000,0.946184,1,8
8,CUST1010,Kite Retail Ltd,Small Retailer,Bristol,Electronics,Sarah Chen,6,17.28,-0.500000,-0.084257,0.333333,0.333333,0.000000,0.333333,0.333333,0.925687,1,9
9,CUST1073,Blue Commerce Ltd,Small Retailer,Glasgow,Food & Drink,Aisha Rahman,10,50.37,-0.571429,-0.501339,0.200000,0.000000,0.000000,0.300000,0.000000,0.887885,1,10



## 15. Short business interpretation template

Use this wording in your model summary:

> The model uses the previous 60 days of customer delivery behaviour to predict whether that customer will reduce business in the next 60 days. The strongest predictors should be interpreted as early warning indicators, not proven causes. The watchlist should therefore be used by account managers for targeted review and intervention, not as an automated decision system.


In [94]:

print("Files created:")
print("- customer_60day_model_table.csv")
print("- threshold_results.csv")
print("- feature_importance.csv")
print("- churn_watchlist.csv")


Files created:
- customer_60day_model_table.csv
- threshold_results.csv
- feature_importance.csv
- churn_watchlist.csv
